In [8]:
# importing the packages
import numpy as np
import pandas as pd
from itertools import chain
from ortools.linear_solver import pywraplp
from warnings import filterwarnings

filterwarnings("ignore")

# # create a zero np_array
# pair_matrix = np.zeros((len(pairings), len(np_arr)))

# # fill the values of the flight legs in each pairing with 1
# for i, pair in enumerate(pairings):
#     pair_matrix[i, list(chain.from_iterable(pair))] = 1

In [9]:
# Reading the dataframe and converting it to a numpy array
np_arr = pd.read_csv(
    "../data/flight_legs/data.csv",
    parse_dates=["start_time", "end_time"],
).to_numpy()


# read the duties from the txt file
with open("../data/pairings/pairings.txt") as file:
    pairings = [list(chain.from_iterable(eval(line))) for line in file]


# calculate the cost matrix from pair list
cost_matrix = np.array(
    [
        (np_arr[pair[-1]][4] - np_arr[pair[0]][3]).total_seconds() / 3600
        for pair in pairings
    ]
).reshape(-1, 1)

# determining the number of flights and tasks
num_pairs = len(pairings)
num_flights = len(np_arr)
print(num_pairs, num_flights)

64924 168


In [10]:
# Initializing the MIP Solver
solver = pywraplp.Solver.CreateSolver("GLOP")

# creating the binary allocation variable
x = np.array([solver.BoolVar("") for i in range(num_pairs)]).reshape(-1, 1)
# declaring the constraints
# Uniqueness of flight legs and all flight legs covered
for i in range(num_flights):
    solver.Add(
        solver.Sum([x[j][0] * 1.0 for j in range(num_pairs) if i in pairings[j]]) == 1.0
    )
# Objective function
solver.Minimize(sum(cost_matrix[i][0] * x[i][0] for i in range(num_pairs)))

# Solve the problem
status = solver.Solve()

# print the state of the optimization problem
if status == pywraplp.Solver.OPTIMAL:
    print("The Solution is OPTIMAL")
elif status == pywraplp.Solver.FEASIBLE:
    print("The Solution is Feasible")
else:
    print("The Solution is not Feasible")

x_values = [x[i][0].solution_value() for i in range(num_pairs)]
print(len(x_values))
index = [i for i in range(num_pairs) if x_values[i] > 0]
print(index)
print(solver.Objective().Value())
# Access the dual values (shadow prices)
dual_values = [constraint.dual_value() for constraint in solver.constraints()]
print(dual_values)

The Solution is not Feasible
64924
[]
0.0
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [4]:
# Initializing the MIP Solver
solver = pywraplp.Solver.CreateSolver("SAT")

# creating the binary allocation variable
x = np.array([solver.BoolVar("") for i in range(num_pairs)]).reshape(-1, 1)
# declaring the constraints
# Uniqueness of flight legs and all flight legs covered
for i in range(num_flights):
    solver.Add(
        solver.Sum(
            [
                x[j][0] * pair_matrix[j][i]
                for j in range(num_pairs)
                if pair_matrix[j][i] != 0.0
            ]
        )
        == 1.0
    )
# Objective function
solver.Minimize(sum(cost_matrix[i][0] * x[i][0] for i in range(num_pairs)))

# Solve the problem
status = solver.Solve()

# print the state of the optimization problem
if status == pywraplp.Solver.OPTIMAL:
    print("The Solution is OPTIMAL")
elif status == pywraplp.Solver.FEASIBLE:
    print("The Solution is Feasible")
else:
    print("The Solution is not Feasible")

x_values = [x[i][0].solution_value() for i in range(num_pairs)]
print(len(x_values))
index = [i for i in range(num_pairs) if x_values[i] > 0]
print(index)
print(solver.Objective().Value())
# Access the dual values (shadow prices)
dual_values = [constraint.dual_value() for constraint in solver.constraints()]
print(dual_values)

NameError: name 'pair_matrix' is not defined

In [5]:
# importing the packages
import numpy as np
from ortools.linear_solver import pywraplp

# Initialize the solver
solver = pywraplp.Solver.CreateSolver("CBC")
use_dual_simplex: True

# Decision Variable
x = [solver.NumVar(0, 1, f"x_{i}") for i in range(num_pairs)]

# Constraints
for j in range(num_flights):
    solver.Add(sum(pair_matrix[i][j] * x[i] for i in range(num_pairs)) - 1 >= 0)

# Objective function
solver.Minimize(sum(cost_matrix[i][0] * x[i] for i in range(num_pairs)))

# Solve the problem
status = solver.Solve()

# print the state of the optimization problem
x_values = [x[i].solution_value() for i in range(num_pairs)]
print(len(x_values))
index = [i for i in range(num_pairs) if x_values[i] > 0]
print(index)
print(solver.Objective().Value())
# Access the dual values (shadow prices)
dual_values = [constraint.dual_value() for constraint in solver.constraints()]
print(dual_values)

NameError: name 'pair_matrix' is not defined

In [6]:
# importing the packages
import numpy as np
from ortools.sat.python import cp_model

# Declare the CP_SAT model
model = cp_model.CpModel()

# create the decision variable
x = []
for i in range(num_pairs):
    x.append(model.NewBoolVar(f"x_{i}"))

# Constraints
constraints = []
for j in range(num_flights):
    constraints.append(
        model.Add(sum(pair_matrix[i][j] * x[i] for i in range(num_pairs)) >= 1)
    )

# Objective Function
model.Minimize(sum(cost_matrix[i][0] * x[i] for i in range(num_pairs)))

# Solve the Problem
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL:
    selected_pairs = [i for i in range(num_pairs) if solver.Value(x[i]) == 1]
    print(selected_pairs)
else:
    print("oops")

print(solver.ObjectiveValue(), len(selected_pairs))

NameError: name 'pair_matrix' is not defined

In [4]:
import numpy as np
from scipy.optimize import linprog


def solve_mip(pair_matrix, cost_matrix):
    num_pairs, num_flights = pair_matrix.shape

    # Objective coefficients for the binary decision variables
    c = cost_matrix.flatten().reshape(-1, 1)
    print(c.shape)
    # Coefficients matrix for the constraints (each flight leg should be covered at least once)
    A_eq = np.vstack([pair_matrix, np.ones((1, num_flights))])
    b_eq = np.ones(num_flights + 1)  # RHS for equality constraints

    # Bounds for decision variables (x should be binary)
    bounds = [(0, 1) for _ in range(num_pairs)]
    print(A_eq.shape)
    # Solve the linear programming problem
    result = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")

    # Extract the results
    minimized_cost = result.fun
    selected_pairs = np.round(result.x).astype(int)
    dual_variables = result.slack[
        :-1
    ]  # Slack variables correspond to dual variables in equality constraints

    return minimized_cost, selected_pairs, dual_variables


minimized_cost, selected_pairs, dual_variables = solve_mip(pair_matrix, cost_matrix)

# Print the results
print("Minimized Cost:", minimized_cost)
print("Selected Pairs:", selected_pairs)
print("Optimal Dual Variables:", dual_variables)

(80706, 1)
(80707, 240)


ValueError: Invalid input for linprog: A_eq must have exactly two dimensions, and the number of columns in A_eq must be equal to the size of c